# 18 — Cored vs coreless elliptical? & the impact on our Sérsic fits

**Question.** AGEL J020613−011417's deflector has σ_e = 267 km/s, log M⋆ ≈ 11.5, passive, and sits
~100 kpc from cluster ACT-CL J0206 (a BCG-like environment). By the core/coreless dichotomy
(Faber+1997, Lauer+2007, Kormendy+2009) a galaxy this massive (σ ≳ 240) is **expected to be a *core*
(cored) elliptical** — a central surface-brightness *deficit* from binary-SMBH core scouring.

**Two coupled sub-questions:**
1. **Is there a resolvable core?** Expected core radius r_b ~ 0.1–0.3 kpc → **0.014–0.041″** at z=0.676,
   *below* every PSF we have (JWST SW ~0.05″, F200LP ~0.07″, F140W ~0.13″). So a core is likely
   **unresolved** and contributes <1% of the light — negligible for M⋆, and the M_BH–core-scouring
   cross-check is not feasible.
2. **Is our single-Sérsic fit trustworthy?** Our table fit gives **n ≈ 1.2–1.6** — anomalously *low*
   for a massive (likely cD) elliptical (expect n ≈ 4–8; Caon+1993). If that is a **box-limited** fit
   (fit box ≈ 1.5 r_eff misses the extended high-n envelope), the total-light extrapolation — and hence
   **M⋆ / the aperture correction** — would be **underestimated**. *This* is where "cored vs coreless"
   actually bites our measurement.

**Tests in this notebook:**
- **A — box-size / Sérsic-n sensitivity** (does n & total light keep rising with box → box-limited?).
- **B — 1D radial SB profile + inner residual** (central deficit=core vs excess=nuclear disk; PSF-caveated).
- **C — core-Sérsic vs single-Sérsic** (framework; likely inconclusive — core unresolved).
- PSF caveat + synthesis.

Env: `ISMgas`. Reuses `scripts.mask_method_comparison.load_band` + a relaxed-bounds Sérsic fitter.

In [1]:
%matplotlib inline
import os, sys
# Anchor at the repo root: scripts AND their internal data paths are relative to it.
if os.path.isdir('../scripts') and not os.path.isdir('scripts'):
    os.chdir('..')
sys.path.insert(0, os.path.abspath('.'))
import numpy as np, json
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.modeling.models import Sersic2D
from astropy.modeling.fitting import LevMarLSQFitter
from scipy.special import gammaincinv, gamma
from scripts.mask_method_comparison import load_band
from scripts.sersic_total_photometry import moment_seed

REG = json.load(open('results/PAPER_VALUES.json'))
KPC = REG['constants']['kpc_per_arcsec']['value']    # 7.276 kpc/arcsec at z=0.676
print('scale = %.3f kpc/arcsec; sigma_e=%.0f, logM*=%.2f' % (
    KPC, REG['sigma_e']['central']['value'], REG['logMstar']['central_10pct']['value']))

ORDER = ['F200LP', 'F140W', 'F150W2', 'F322W2']
# expert L3 deflector masks (arc + interlopers), same as scripts/sersic_total_photometry.py
EXP = {'F200LP':'../velocity_dispersion_from_IFU/AGEL020613-011417A_F200LP_WFC3_cutout_L3_mask.fits',
       'F140W':'../velocity_dispersion_from_IFU/AGEL020613-011417A_F140W_WFC3_cutout_L3_mask.fits',
       'F150W2':'../velocity_dispersion_from_IFU/jw05594-o101_t103_nircam_clear-f150w2_i2d_mask.fits',
       'F322W2':'../velocity_dispersion_from_IFU/jw05594-o101_t103_nircam_clear-f322w2_i2d_mask.fits'}
BANDS = {n: load_band(n) for n in ORDER}
MASKS = {n: fits.getdata(EXP[n]).astype(bool) for n in ORDER}
for n in ORDER:
    print('%-7s pix=%.3f"  shape=%s' % (n, BANDS[n]['pix_scale'], BANDS[n]['img'].shape))


def fit_sersic_box(b, mask, box_arcsec, n_max=6.0):
    # Single-Sérsic fit inside +/-box_arcsec, masked; sky from the outer 20% ring; multi-start.
    # Bounds: n in (0.5, n_max); r_eff in (0.3", half-box) — capping r_eff at half the box stops the
    # runaway to a giant low-SB ICL/field component that destabilises large-box fits. 'railed' flags a
    # fit pinned at a bound (unreliable). Returns dict(n, r_eff_as, ellip, theta, m_tot, amp, model, off, sky, railed).
    ps, xc, yc, zp = b['pix_scale'], b['cx'], b['cy'], b['ab_zp']
    half = int(np.ceil(box_arcsec/ps))
    y1,y2 = max(0,int(yc)-half), min(b['img'].shape[0],int(yc)+half+1)
    x1,x2 = max(0,int(xc)-half), min(b['img'].shape[1],int(xc)+half+1)
    sub = np.nan_to_num(b['img'][y1:y2,x1:x2].astype(float)); sm = mask[y1:y2,x1:x2]
    yy,xx = np.mgrid[:sub.shape[0],:sub.shape[1]]; cx0,cy0 = xc-x1, yc-y1
    r = np.hypot(xx-cx0,yy-cy0)*ps
    ring = (r>0.8*box_arcsec) & ~sm
    sky = float(np.median(sub[ring])) if ring.any() else 0.0
    sub = sub - sky; w = (~sm).astype(float)
    ci,cj = int(np.clip(round(cy0),0,sub.shape[0]-1)), int(np.clip(round(cx0),0,sub.shape[1]-1))
    peak = float(np.median(sub[max(0,ci-1):ci+2, max(0,cj-1):cj+2])) or 1.0
    reff0 = 2.0/ps
    bnds = {'n':(0.5,n_max), 'r_eff':(0.3/ps, 4.5/ps), 'ellip':(0.,0.7),
            'amplitude':(peak*1e-3, peak*1.5), 'x_0':(cx0-3,cx0+3), 'y_0':(cy0-3,cy0+3)}
    e_mom,th_mom = moment_seed(sub,w,cx0,cy0,ps,3.0)
    starts=[(e_mom,th_mom)]+[(0.4,t) for t in np.linspace(0,np.pi,4,endpoint=False)]+[(0.0,0.0)]
    best,br=None,np.inf
    for e0,th0 in starts:
        m0=Sersic2D(amplitude=peak*0.05,r_eff=reff0*0.6,n=3.0,x_0=cx0,y_0=cy0,
                    ellip=min(e0,0.69),theta=th0,bounds=bnds)
        c=LevMarLSQFitter()(m0,xx,yy,sub,weights=w,maxiter=2000)
        rss=float(np.sum((w*(sub-c(xx,yy)))**2))
        if rss<br: best,br=c,rss
    n=float(best.n.value); reff_px=float(best.r_eff.value); el=float(best.ellip.value)
    bn=gammaincinv(2*n,0.5)
    Ftot=2*np.pi*float(best.amplitude.value)*reff_px**2*n*np.exp(bn)/bn**(2*n)*gamma(2*n)*(1-el)
    mtot=-2.5*np.log10(Ftot)+zp if Ftot>0 else np.nan
    railed = (n<=0.52) or (n>=n_max-0.05) or (reff_px*ps>=4.4)
    return dict(n=n, r_eff_as=reff_px*ps, ellip=el, theta=float(best.theta.value),
                m_tot=mtot, amp=float(best.amplitude.value), model=best, off=(x1,y1), sky=sky, railed=railed)
print('helpers ready')

scale = 7.276 kpc/arcsec; sigma_e=267, logM*=11.46


Set DATE-AVG to '2024-08-27T06:21:29.964' from MJD-AVG.
Set DATE-END to '2024-08-27T06:40:22.696' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -15.598244 from OBSGEO-[XYZ].
Set OBSGEO-H to 1415448491.890 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2024-08-27T06:21:29.992' from MJD-AVG.
Set DATE-END to '2024-08-27T06:40:22.760' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -15.598244 from OBSGEO-[XYZ].
Set OBSGEO-H to 1415448491.890 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


F200LP  pix=0.050"  shape=(500, 500)
F140W   pix=0.080"  shape=(312, 312)
F150W2  pix=0.031"  shape=(4702, 4720)
F322W2  pix=0.063"  shape=(2256, 2262)
helpers ready


## Test A — box-size / Sérsic-n sensitivity  *(the M⋆-critical test)*

Re-fit a single Sérsic in each band over growing boxes. If the galaxy were well-described by a single
Sérsic, **n, r_eff, and m_tot would plateau**. If instead they **keep rising** (n → 4–8, m_tot
brightening), the fit is **box-limited** — it's chasing an extended high-n / cD envelope, and the
small-box total-light we used for M⋆ is an **underestimate**.

In [2]:
# HST cutouts are small & clean -> box test stable to ~10"; JWST are large mosaics -> field/ICL
# sources contaminate beyond ~6", so restrict JWST to small boxes. Railed fits are flagged & excluded.
BOXES = {'F200LP':[3,4,5,6,8,10], 'F140W':[3,4,5,6,8,10], 'F150W2':[3,4,5,6], 'F322W2':[3,4,5,6]}
resA = {n: [] for n in ORDER}
for n in ORDER:
    b, m = BANDS[n], MASKS[n]
    maxbox = (min(b['img'].shape)//2 - 2) * b['pix_scale']
    BOX = [x for x in BOXES[n] if x <= maxbox]
    print('\n%s' % n); print('%6s %6s %8s %8s  %s' % ('box', 'n', 'r_eff', 'm_tot', 'flag'))
    for box in BOX:
        r = fit_sersic_box(b, m, box)
        resA[n].append((box, r['n'], r['r_eff_as'], r['m_tot'], r['railed']))
        print('%6.0f %6.2f %8.2f %8.2f  %s' % (box, r['n'], r['r_eff_as'], r['m_tot'],
                                               'RAILED (unreliable)' if r['railed'] else ''))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for n in ORDER:
    a = np.array([(bx,nn,re,mt) for bx,nn,re,mt,rl in resA[n] if not rl])  # plot only stable fits
    if len(a)==0: continue
    ax[0].plot(a[:,0], a[:,1], 'o-', label=n); ax[1].plot(a[:,0], a[:,3], 'o-', label=n)
ax[0].axhspan(4, 8, color='gray', alpha=0.15, label='expected n (massive E)')
ax[0].set(xlabel='fit box (arcsec)', ylabel='Sérsic n', title='n vs box — rising & not plateauing ⇒ box-limited')
ax[1].set(xlabel='fit box (arcsec)', ylabel='total mag (AB)', title='total light vs box — brightening ⇒ small-box underestimate')
ax[1].invert_yaxis()
for a_ in ax: a_.legend(fontsize=8); a_.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('results/figures/cored_test_A_boxsweep.png', dpi=130, bbox_inches='tight'); plt.show()
# headline delta on STABLE fits only: table-box (4") vs largest stable box
print('\nΔm_tot (4" -> largest stable box), mag  [+ = more light at larger box]:')
for n in ORDER:
    st = [(bx,nn,re,mt) for bx,nn,re,mt,rl in resA[n] if not rl]
    if not st: print('  %-7s (no stable fit)' % n); continue
    a=np.array(st); row4=a[a[:,0]==4]
    if len(row4)==0: continue
    print('  %-7s %+.2f mag  (n: %.2f -> %.2f over box 4->%.0f")' % (
        n, row4[0,3]-a[-1,3], row4[0,1], a[-1,1], a[-1,0]))


F200LP
   box      n    r_eff    m_tot  flag
     3   0.99     1.45    21.54  


     4   1.14     1.72    21.16  
     5   0.50     1.54    21.19  RAILED (unreliable)


     6   0.50     1.66    21.06  RAILED (unreliable)


     8   0.50     1.75    20.98  RAILED (unreliable)


    10   0.50     1.77    20.96  RAILED (unreliable)

F140W
   box      n    r_eff    m_tot  flag
     3   0.91     1.20    18.97  


...

     6   0.50     4.50    14.31  RAILED (unreliable)

F322W2
   box      n    r_eff    m_tot  flag


     3   0.95     1.13    18.41  
     4   1.50     1.46    17.98  


     5   1.83     1.90    17.68  


     6   1.96     2.25    17.55  

Δm_tot (4" -> largest stable box), mag  [+ = more light at larger box]:
  F200LP  +0.00 mag  (n: 1.14 -> 1.14 over box 4->4")
  F140W   +0.73 mag  (n: 1.17 -> 1.98 over box 4->10")
  F150W2  (no stable fit)
  F322W2  +0.43 mag  (n: 1.50 -> 1.96 over box 4->6")


/var/folders/mm/fytlrh2s5nx_x7cvgsdllc2h0000gr/T/ipykernel_43724/3962662947.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig('results/figures/cored_test_A_boxsweep.png', dpi=130, bbox_inches='tight'); plt.show()


**Read-off.** If Δm_tot ≳ 0.3 mag and n climbs toward ~3–5 without plateauing, the single-Sérsic
total is box-limited → **M⋆ (built on the 4″ box) is underestimated by ~0.4×Δm_tot dex** and, worse,
is **not well-defined** (the cD envelope blends into the ICL). *Caveat:* at large box the sky ring and
any residual ICL/neighbour light inflate the total — the brightening is real but the absolute
large-box total needs an ICL-aware decomposition, not a single Sérsic.

## Test A2 — empirical curve-of-growth total light *(quantifies the M⋆ underestimate)*

Test A says the single-Sérsic *total* is box-limited. Here we measure the **empirical** enclosed light
directly — cumulative flux in growing elliptical apertures (masked, sky-subtracted, validated-Sérsic-
filled inside the mask) — and compare to the single-Sérsic total. Driver: `scripts/cog_total_light.py`
→ `results/cog_total_light.npz`, `results/figures/cog_total_light.png`.

**Result (2026-06-16):** at a = 8″ the CoG is **0.44–0.56 mag brighter** than the single-Sérsic total in
the three redder bands (F200LP rest-UV is sky-noise dominated, discarded), and **still rising** (no
convergence → cD envelope/ICL). Feeding the CoG@8″ magnitudes to the quiescent-prior Bagpipes fit gives
**log M⋆ = 11.68 +0.06/−0.07**, i.e. **+0.22 dex above the single-Sérsic headline (11.46)**. This lands
only **+0.04 dex beyond the existing +1σ upper bound (11.65)** → the total-light/cD-envelope ambiguity is
**already captured** by the systematic budget (the apcorr-model term spans the more-outer-light
direction). So the CoG is kept as a **cross-check, not a separate systematic** (adding +0.22 would
double-count). CoG caveats: non-convergent (sky/ICL) and strongly bandpass-dependent. See Synthesis.

In [3]:
# Empirical curve-of-growth total light vs the single-Sérsic total (per band).
from scripts.cog_total_light import cog_band, ORDER as _CO
cog = {n: cog_band(n) for n in _CO}
fig, ax = plt.subplots(figsize=(7,5))
print('%-7s %8s %8s %9s' % ('band','Sersic','CoG@8"','ΔCoG-Ser'))
for n in _CO:
    r = cog[n]; ax.plot(r['aa'], r['mcum'], '-', label='%s CoG'%n)
    ax.axhline(r['mser'], ls=':', color=ax.lines[-1].get_color(), alpha=0.6)
    print('%-7s %8.2f %8.2f %+9.2f' % (n, r['mser'], r['m8'], r['m8']-r['mser']))
ax.invert_yaxis(); ax.set_xlabel('elliptical a (arcsec)'); ax.set_ylabel('cumulative mag (AB)')
ax.set_title('CoG (solid) vs single-Sérsic total (dotted) — CoG keeps rising ⇒ Sérsic underestimates')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('results/figures/cog_total_light.png', dpi=130, bbox_inches='tight'); plt.show()
print('\\nQuiescent-prior Bagpipes on CoG@8" mags -> log M* = 11.68 (+0.22 dex vs single-Sérsic 11.46).')

Set DATE-AVG to '2024-08-27T06:21:29.964' from MJD-AVG.
Set DATE-END to '2024-08-27T06:40:22.696' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -15.598244 from OBSGEO-[XYZ].
Set OBSGEO-H to 1415448491.890 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


Set DATE-AVG to '2024-08-27T06:21:29.992' from MJD-AVG.
Set DATE-END to '2024-08-27T06:40:22.760' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -15.598244 from OBSGEO-[XYZ].
Set OBSGEO-H to 1415448491.890 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


band      Sersic   CoG@8"  ΔCoG-Ser
F200LP     20.91    20.44     -0.48
F140W      18.51    18.04     -0.47
F150W2     18.36    16.23     -2.13
F322W2     17.95    16.56     -1.39
\nQuiescent-prior Bagpipes on CoG@8" mags -> log M* = 11.68 (+0.22 dex vs single-Sérsic 11.46).


/var/folders/mm/fytlrh2s5nx_x7cvgsdllc2h0000gr/T/ipykernel_43724/465685427.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig('results/figures/cog_total_light.png', dpi=130, bbox_inches='tight'); plt.show()


## Test B — 1D radial surface-brightness profile + inner residual

Azimuthally-averaged (elliptical) SB profile of the data vs the best Sérsic model. A **central deficit**
(data below model inside ~r_b) ⇒ core; a **central excess** ⇒ nuclear disk/extra light.
**PSF caveat:** our fits use *no* PSF (env lacks webbpsf), and the expected core (0.01–0.04″) is below
the PSF — so the inner ~1–2 PSF FWHM is unreliable; treat this as qualitative.

In [4]:
def ellip_profile(b, mask, model, off, sky, ellip, theta, ps, rmax_as=6.0, nbin=22):
    img = np.nan_to_num(b['img']) - sky
    x1, y1 = off
    yy, xx = np.mgrid[:img.shape[0], :img.shape[1]]
    cx, cy = b['cx'], b['cy']
    ct, st = np.cos(-theta), np.sin(-theta)
    xr = (xx-cx)*ct - (yy-cy)*st; yr = (xx-cx)*st + (yy-cy)*ct
    q = 1 - ellip
    rell = np.hypot(xr, yr/max(q,0.2)) * ps          # elliptical radius (arcsec)
    mod = np.clip(np.asarray(model(xx - x1, yy - y1), float), 0, None)
    edges = np.linspace(0, rmax_as, nbin+1); rc = 0.5*(edges[:-1]+edges[1:])
    d_sb, m_sb = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        sel = (rell>=lo) & (rell<hi) & ~mask & np.isfinite(img)
        d_sb.append(np.median(img[sel]) if sel.sum()>3 else np.nan)
        sel2 = (rell>=lo) & (rell<hi)
        m_sb.append(np.median(mod[sel2]) if sel2.sum()>0 else np.nan)
    return rc, np.array(d_sb), np.array(m_sb)

PSF = {'F200LP':0.07, 'F140W':0.13, 'F150W2':0.05, 'F322W2':0.13}
fig, axes = plt.subplots(2, 4, figsize=(16, 6), gridspec_kw={'height_ratios':[3,1]})
for j, n in enumerate(ORDER):
    b, m = BANDS[n], MASKS[n]; ps = b['pix_scale']; zp = b['ab_zp']
    r = fit_sersic_box(b, m, 10.0)                       # reference model (large box)
    rc, d, mo = ellip_profile(b, m, r['model'], r['off'], r['sky'], r['ellip'], r['theta'], ps)
    pa = lambda f: -2.5*np.log10(np.clip(f,1e-12,None))/ps**0 + zp + 2.5*np.log10(ps**2)  # mag/arcsec^2
    d_mu, m_mu = pa(d), pa(mo)
    ax, axr = axes[0,j], axes[1,j]
    ax.plot(rc, d_mu, 'k.-', label='data'); ax.plot(rc, m_mu, 'r-', label='Sérsic (box 10")')
    ax.axvspan(0, PSF[n], color='gray', alpha=0.2); ax.invert_yaxis()
    ax.set_title('%s (n=%.1f)' % (n, r['n'])); ax.set_ylabel(r'$\mu$ (mag/arcsec$^2$)'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    axr.axhline(0, color='r'); axr.plot(rc, d_mu-m_mu, 'k.-'); axr.axvspan(0, PSF[n], color='gray', alpha=0.2)
    axr.set_xlabel('elliptical r (arcsec)'); axr.set_ylabel('Δμ'); axr.set_ylim(-0.6,0.6); axr.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('results/figures/cored_test_B_profile.png', dpi=130, bbox_inches='tight'); plt.show()
print('Inner Δμ>0 (data fainter than model) inside the gray PSF band ⇒ central deficit (core-like);')
print('Δμ<0 (data brighter) ⇒ central excess. Below the PSF (gray) this is NOT reliable.')

Inner Δμ>0 (data fainter than model) inside the gray PSF band ⇒ central deficit (core-like);
Δμ<0 (data brighter) ⇒ central excess. Below the PSF (gray) this is NOT reliable.


/var/folders/mm/fytlrh2s5nx_x7cvgsdllc2h0000gr/T/ipykernel_43724/3697294991.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig('results/figures/cored_test_B_profile.png', dpi=130, bbox_inches='tight'); plt.show()


## Test C — core-Sérsic vs single-Sérsic *(PSF-convolved; DONE 2026-06-17)*

Now executed with **real PSFs** (built in `scripts/build_psf_models.py`, A6). A **core-Sérsic** profile
(Graham+2003, Trujillo+2004) adds an inner break radius r_b below which the profile flattens. Driver:
`scripts/core_sersic_test.py` → `results/core_sersic_test.npz`, `results/figures/core_sersic_test.png`.
PSF-convolved 2D fits on F150W2 (finest PSF, 0.049″) and F140W (STScI PSFSTD).

**Result.** A free core-Sérsic "prefers" the fit by ΔBIC = 386 (F150W2) / 43 (F140W) — **but this is a
fitting artifact, not a core detection:**
- the fitted **r_b = 0.50″/0.34″ (3.6/2.5 kpc)** is ~10–50× too large for a depleted core (expected
  r_b ~ 0.005–0.05″ for M•~5×10⁸; Lauer+2007, Rusli+2013) and ~10–20× the PSF HWHM;
- the single-Sérsic inner residual does **NOT deepen toward r=0** (flat ~−6%, oscillating ±5–15%) — the
  signature of a real core is a residual that *deepens* toward the centre, which we do not see;
- the single-Sérsic n rails low (0.9–1.1 vs validated 1.2–1.6) — so the core-Sérsic's inner flattening is
  absorbing the **global** single-Sérsic mismatch (= the outer cD envelope of Test A), not a nucleus.

**Verdict.** A genuine depleted core is **UNRESOLVED** at every PSF (finest HWHM 0.024″ = 0.17 kpc) →
we **cannot distinguish cored vs coreless**. Upper limit **r_b < 0.024″ (0.17 kpc)**. Single-Sérsic is the
pragmatic inner model; a depleted core removes <1% of the light → **negligible for M⋆**.

> ⚠️ **Methodological caution:** a free core-Sérsic ΔBIC "preference" is *not* a core detection when the
> baseline single-Sérsic mis-fits the galaxy globally. Always check that (a) r_b is both resolved
> (> PSF HWHM) *and* physically small (< ~0.3 kpc), and (b) the inner residual deepens toward r=0.

```
# core-Sérsic (Trujillo+2004): I(r) = I_b 2^(-γ/α) [1+(r_b/r)^α]^(γ/α)
#                                     * exp{ -b_n [ (r^α + r_b^α)/r_e^α ]^(1/(αn)) }
# Implemented PSF-convolved in scripts/core_sersic_test.py (α=10 fixed; r_b, γ free).
```

## PSF models (prerequisite for inner-profile work) — DONE 2026-06-17 (A6)

Earlier these fits used **no PSF** (`ISMgas` lacks `webbpsf`/TinyTim). PSFs are now built **in-env**,
without a synthetic generator (`scripts/build_psf_models.py` → `results/psf_models/<band>_psf.npz`):
- **F140W:** STScI `PSFSTD_WFC3IR_F140W.fits` (4×-oversampled empirical library), FWHM **0.103″**.
- **F200LP / F150W2 / F322W2:** empirical EPSF (photutils `EPSFBuilder`) from isolated, unsaturated
  field stars — HST from the **full-frame** drc/drz (146 / 102 stars; the 25″ lens cutout has none),
  JWST from the i2d mosaic (3 SW / 13 LW). FWHM **0.085″ / 0.049″ / 0.112″** — all match instrument values.

`scripts/psf_star_census.py` is the feasibility census. These PSFs drive the PSF-convolved Test C above.
The finest PSF (F150W2, HWHM 0.024″) still cannot resolve the expected depleted core (r_b ~ 0.005–0.05″).

## Synthesis (updated 2026-06-17)

- **Cored?** Physically **expected** (σ=267 ≫ 240 divide; massive cD near ACT-CL J0206). Observationally
  the core (r_b ~ 0.005–0.05″) is **below every PSF** → **unresolved** (Test C, A7: PSF-convolved
  core-Sérsic returns r_b=2.5–3.6 kpc with no inner deepening = a global-fit artifact, NOT a nucleus;
  upper limit r_b < 0.024″ = 0.17 kpc). <1% of the light; the M_BH–core-scouring cross-check is not
  feasible. ⇒ a single Sérsic is the pragmatic *inner* light model; the core, if present, can't be
  measured here. **PSF models now exist for all 4 bands (A6)** — the limit is resolution, not tooling.
- **The real impact is on the TOTAL light, not the core.** Tests A & A2 show the single-Sérsic total is
  **box-limited and non-convergent** (n & cumulative light keep rising; CoG@8″ is +0.44–0.56 mag brighter
  than the Sérsic total and still climbing). ⇒ the headline **M⋆ = 11.46 is a LOWER BOUND**; the empirical
  curve-of-growth total gives **log M⋆ ≈ 11.68 (+0.22 dex)** and is itself a lower bound (envelope/ICL).
- **DECISION (2026-06-17):** central M⋆ stays the **single-Sérsic 11.46** — chosen on *measurement*
  grounds (well-defined & reproducible: fixed aperture + a stated correction), NOT relation-consistency
  (we are building an *independent* probe; the local relations are guide-the-eye only). The CoG total
  (11.69) is recorded as a **cross-check**: it lands only **+0.04 dex beyond the existing +1σ upper
  bound (11.65)**, so the total-light/cD-envelope ambiguity is **already captured** by the systematic
  budget — the apcorr-model term (auto more-extended Sérsic = the more-outer-light direction) spans it.
  A *separate* one-sided +0.22 term would **double-count** → NOT added. CoG caveats reinforcing this:
  non-convergent (sky/ICL) and strongly bandpass-dependent (envelope colour ⇒ uncertain extra-light M/L).
- Net M⋆ headline unchanged: **11.5 ± 0.1 (stat) ± 0.2 (sys)**; CoG 11.69 in the registry as cross-check.